In [ ]:
%pip -q install -U openai pandas scipy scikit-learn tqdm


In [ ]:
from google.colab import drive, userdata
from IPython.display import display
from pathlib import Path
from datetime import datetime, timezone
import base64
import hashlib
import json
import os
import re
import time

import numpy as np
import pandas as pd
from scipy.stats import spearmanr
from sklearn.metrics import mean_absolute_error
from tqdm.auto import tqdm
from openai import OpenAI

                                               
RUN_DEV_API_CALLS = True
RUN_TEST_API_CALLS = False
RUN_LIMIT = None                                         

MODEL = 'gpt-5.5'
REASONING_EFFORT = 'none'
QUERY_IMAGE_DETAIL = 'original'
DEMONSTRATION_IMAGE_DETAIL = 'low'
REQUEST_SLEEP_SECONDS = 0.2
MAX_RETRIES = 3

                                                                                  
                                                                   
DEMONSTRATION_IDS = [
    'img_004_v3',                    
    'img_023_v5',                          
    'img_035_v4',                     
    'img_027_v4',                          
    'img_010_v2',                    
    'img_024_v2',                          
]

drive.mount('/content/drive')
IMAGEEVAL_ROOT = Path('/content/drive/MyDrive/Dr. Lulwah - Ahmed/ImageEVAl')
PROJECT_DIR = IMAGEEVAL_ROOT / 'ImageEval2026_Task2_CRAI_Bench'
DATA_ROOT_CANDIDATES = [
    PROJECT_DIR / 'data',
    IMAGEEVAL_ROOT / 'train_dev',
]

EXPERIMENT_ROOT = PROJECT_DIR / 'cea_direct_fewshot_v1'
CACHE_DIR = EXPERIMENT_ROOT / 'cache'
OUTPUT_DIR = EXPERIMENT_ROOT / 'outputs'
CACHE_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

client = None

def get_openai_client():
    global client
    if client is None:
        key = userdata.get('openai')
        if not key:
            raise RuntimeError('Colab secret "openai" is missing.')
        os.environ['OPENAI_API_KEY'] = key
        client = OpenAI()
    return client

pd.set_option('display.max_columns', 100)
pd.set_option('display.max_colwidth', 140)

print('Project:', PROJECT_DIR)
print('Experiment:', EXPERIMENT_ROOT)
print('Dev API calls:', RUN_DEV_API_CALLS)
print('Test API calls:', RUN_TEST_API_CALLS)


In [ ]:
def parse_base_id(instance_id: str) -> str:
    return re.sub(r'_v\d+$', '', str(instance_id))

def find_split_dir(split: str) -> Path:
    for root in DATA_ROOT_CANDIDATES:
        candidate = root / split
        if (candidate / 'captions.tsv').exists():
            return candidate
    searched = [str(root / split) for root in DATA_ROOT_CANDIDATES]
    raise FileNotFoundError(f'Could not find {split} data. Searched: {searched}')

def find_image(folder: Path, stem: str) -> Path:
    for suffix in ['.png', '.jpg', '.jpeg', '.webp']:
        path = folder / f'{stem}{suffix}'
        if path.exists():
            return path
    raise FileNotFoundError(f'Image {stem} not found in {folder}')

def load_split(split: str, require_human_gold: bool) -> pd.DataFrame:
    split_dir = find_split_dir(split)
    captions = pd.read_csv(split_dir / 'captions.tsv', sep='\t')
    frame = captions.copy()
    human_path = split_dir / 'gold_human.tsv'
    if human_path.exists():
        human = pd.read_csv(human_path, sep='\t')
        keep = ['id', 'CRAI_CEA']
        frame = frame.merge(human[keep], on='id', how='left', validate='one_to_one')
    elif require_human_gold:
        raise FileNotFoundError(human_path)

    frame['id'] = frame['id'].astype(str)
    frame['base_id'] = frame['id'].map(parse_base_id)
    frame['ref_image_path'] = frame['base_id'].map(
        lambda value: str(find_image(split_dir / 'imgs' / 'ref', value))
    )
    frame['generated_image_path'] = frame['id'].map(
        lambda value: str(find_image(split_dir / 'imgs' / 'generated', value))
    )
    if not frame['id'].is_unique:
        raise ValueError(f'{split}: duplicate IDs')
    return frame

train_df = load_split('train', require_human_gold=True)
dev_df = load_split('dev', require_human_gold=True)
try:
    test_df = load_split('test', require_human_gold=False)
except FileNotFoundError:
    test_df = None

if set(train_df['base_id']) & set(dev_df['base_id']):
    raise ValueError('Train/dev reference groups overlap')

demo_rows = train_df.set_index('id').loc[DEMONSTRATION_IDS].reset_index()
if len(demo_rows) != len(DEMONSTRATION_IDS):
    raise ValueError('A demonstration ID is missing')
if demo_rows['CRAI_CEA'].isna().any():
    raise ValueError('A demonstration is missing its human CEA label')
if set(demo_rows['base_id']) & set(dev_df['base_id']):
    raise ValueError('A demonstration group overlaps dev')

print('Train:', len(train_df), '| Dev:', len(dev_df), '| Test:', 0 if test_df is None else len(test_df))
display(demo_rows[['id', 'category', 'CRAI_CEA']])


In [ ]:
PROMPT_VERSION = 'direct-fewshot-cea-v1'

DIRECT_CEA_PROMPT = r"""
You are evaluating Cultural Element Accuracy (CEA) for CRAI-Bench.

You receive:
1. an authentic Qatari reference image,
2. the current caption used to generate an image, and
3. the generated image.

CEA asks: Are the expected cultural elements present and correctly depicted?

Use the labelled examples to follow the human scoring scale. Predict one continuous
score from 0.0 to 1.0. Evaluate CEA only; do not score contextual coherence,
cultural specificity, cultural integrity, hallucination penalty, aesthetics, or
general image quality as separate factors.

Return only this JSON object:
{"CRAI_CEA": <number from 0.0 to 1.0>}
"""

def prompt_hash(text: str, length: int = 12) -> str:
    return hashlib.sha256(text.strip().encode('utf-8')).hexdigest()[:length]

def file_sha256(path: str) -> str:
    digest = hashlib.sha256()
    with Path(path).open('rb') as handle:
        for chunk in iter(lambda: handle.read(1024 * 1024), b''):
            digest.update(chunk)
    return digest.hexdigest()

def demonstration_signature() -> str:
    payload = []
    for row in demo_rows.to_dict('records'):
        payload.append({
            'id': row['id'],
            'gold_cea': float(row['CRAI_CEA']),
            'reference_sha256': file_sha256(row['ref_image_path']),
            'generated_sha256': file_sha256(row['generated_image_path']),
        })
    return prompt_hash(json.dumps(payload, sort_keys=True), length=16)

def cache_tag() -> str:
    return (
        f'{PROMPT_VERSION}_{MODEL}_reasoning-{REASONING_EFFORT}_'
        f'query-{QUERY_IMAGE_DETAIL}_demo-{DEMONSTRATION_IMAGE_DETAIL}_'
        f'prompt-{prompt_hash(DIRECT_CEA_PROMPT)}_demos-{demonstration_signature()}'
    )

print('\nCache configuration:', cache_tag())


In [ ]:
def image_to_data_url(path: str) -> str:
    path = Path(path)
    media = {
        '.png': 'image/png',
        '.jpg': 'image/jpeg',
        '.jpeg': 'image/jpeg',
        '.webp': 'image/webp',
    }.get(path.suffix.lower())
    if media is None:
        raise ValueError(f'Unsupported image type: {path}')
    payload = base64.b64encode(path.read_bytes()).decode('ascii')
    return f'data:{media};base64,{payload}'

def parse_json_object(text: str) -> dict:
    text = str(text).strip()
    text = re.sub(r'^```(?:json)?\s*', '', text, flags=re.I)
    text = re.sub(r'\s*```$', '', text)
    start, end = text.find('{'), text.rfind('}')
    if start < 0 or end < start:
        raise ValueError('No JSON object in response')
    value = json.loads(text[start:end + 1])
    score = float(value['CRAI_CEA'])
    if not np.isfinite(score) or not 0.0 <= score <= 1.0:
        raise ValueError(f'Invalid CRAI_CEA: {score}')
    return {'CRAI_CEA': score}

def user_content(row: pd.Series, detail: str, labelled: bool) -> list[dict]:
    label = 'LABELLED TRAINING EXAMPLE' if labelled else 'INSTANCE TO SCORE'
    text = f"""{label}
Instance ID: {row['id']}
Current caption:
{row['caption']}

The first image is the authentic reference. The second image is the generated image.
Return the CEA JSON only."""
    return [
        {'type': 'input_text', 'text': text},
        {'type': 'input_text', 'text': 'REFERENCE IMAGE:'},
        {'type': 'input_image', 'image_url': image_to_data_url(row['ref_image_path']), 'detail': detail},
        {'type': 'input_text', 'text': 'GENERATED IMAGE:'},
        {'type': 'input_image', 'image_url': image_to_data_url(row['generated_image_path']), 'detail': detail},
    ]

def demonstration_messages() -> list[dict]:
    messages = []
                                                                               
                                                                             
                                                                       
    indexed = train_df.set_index('id', drop=False)
    for instance_id in DEMONSTRATION_IDS:
        row = indexed.loc[instance_id]
        messages.append({
            'role': 'user',
            'content': user_content(row, DEMONSTRATION_IMAGE_DETAIL, labelled=True),
        })
        messages.append({
            'role': 'assistant',
            'content': json.dumps({'CRAI_CEA': float(row['CRAI_CEA'])}),
        })
    return messages

def cache_path(split: str) -> Path:
    return CACHE_DIR / f'{split}_{cache_tag()}.jsonl'

def attempts_path(split: str) -> Path:
    return CACHE_DIR / f'{split}_attempts_{cache_tag()}.jsonl'

def load_jsonl(path: Path) -> list[dict]:
    if not path.exists():
        return []
    records = []
    with path.open(encoding='utf-8') as handle:
        for line_number, line in enumerate(handle, start=1):
            if line.strip():
                try:
                    records.append(json.loads(line))
                except json.JSONDecodeError as exc:
                    raise ValueError(f'Malformed cache {path}:{line_number}') from exc
    return records

def append_jsonl(path: Path, record: dict):
    path.parent.mkdir(parents=True, exist_ok=True)
    with path.open('a', encoding='utf-8') as handle:
        handle.write(json.dumps(record, ensure_ascii=False) + '\n')
        handle.flush()

def call_direct_cea(row: pd.Series, split: str) -> dict:
    for attempt in range(1, MAX_RETRIES + 1):
        created = datetime.now(timezone.utc).isoformat()
        try:
            response = get_openai_client().responses.create(
                model=MODEL,
                reasoning={'effort': REASONING_EFFORT},
                input=[
                    {'role': 'developer', 'content': DIRECT_CEA_PROMPT.strip()},
                    *demonstration_messages(),
                    {'role': 'user', 'content': user_content(row, QUERY_IMAGE_DETAIL, labelled=False)},
                ],
            )
            parsed = parse_json_object(response.output_text)
            record = {
                'instance_id': str(row['id']),
                'CRAI_CEA': parsed['CRAI_CEA'],
                'model': MODEL,
                'reasoning_effort': REASONING_EFFORT,
                'query_image_detail': QUERY_IMAGE_DETAIL,
                'demonstration_image_detail': DEMONSTRATION_IMAGE_DETAIL,
                'prompt_version': PROMPT_VERSION,
                'prompt_hash': prompt_hash(DIRECT_CEA_PROMPT),
                'demonstration_ids': DEMONSTRATION_IDS,
                'demonstration_signature': demonstration_signature(),
                'created_utc': created,
            }
            append_jsonl(attempts_path(split), {
                **record,
                'attempt': attempt,
                'raw_response': response.output_text,
                'status': 'valid',
            })
            return record
        except Exception as exc:
            append_jsonl(attempts_path(split), {
                'instance_id': str(row['id']),
                'attempt': attempt,
                'created_utc': created,
                'status': 'invalid',
                'error': repr(exc),
            })
            if attempt == MAX_RETRIES:
                raise
            time.sleep(2 ** attempt)

def validate_record(record: dict):
    expected = {
        'model': MODEL,
        'reasoning_effort': REASONING_EFFORT,
        'query_image_detail': QUERY_IMAGE_DETAIL,
        'demonstration_image_detail': DEMONSTRATION_IMAGE_DETAIL,
        'prompt_version': PROMPT_VERSION,
        'prompt_hash': prompt_hash(DIRECT_CEA_PROMPT),
        'demonstration_ids': DEMONSTRATION_IDS,
        'demonstration_signature': demonstration_signature(),
    }
    for key, value in expected.items():
        if record.get(key) != value:
            raise ValueError(f'Incompatible cache record {record.get("instance_id")}: {key}')
    score = float(record['CRAI_CEA'])
    if not np.isfinite(score) or not 0.0 <= score <= 1.0:
        raise ValueError(f'Invalid cached score: {score}')

def load_or_infer(frame: pd.DataFrame, split: str, allow_calls: bool) -> pd.DataFrame:
    path = cache_path(split)
    records = load_jsonl(path)
    cached = {}
    for record in records:
        validate_record(record)
        key = str(record['instance_id'])
        if key in cached:
            raise ValueError(f'Duplicate cached ID: {key}')
        cached[key] = record

    requested = frame['id'].astype(str).tolist()
    missing = [value for value in requested if value not in cached]
    print(f'{split} cache: {len(requested) - len(missing)}/{len(requested)}')

    if allow_calls:
        rows = frame.head(RUN_LIMIT) if RUN_LIMIT is not None else frame
        for _, row in tqdm(rows.iterrows(), total=len(rows), desc=f'Direct CEA {split}'):
            instance_id = str(row['id'])
            if instance_id not in cached:
                record = call_direct_cea(row, split)
                append_jsonl(path, record)
                cached[instance_id] = record
                time.sleep(REQUEST_SLEEP_SECONDS)

    output = [cached[value] for value in requested if value in cached]
    return pd.DataFrame(output)


In [ ]:
dev_records = load_or_infer(dev_df, 'dev', allow_calls=RUN_DEV_API_CALLS)

if RUN_TEST_API_CALLS and test_df is None:
    raise RuntimeError('RUN_TEST_API_CALLS=True, but no test split was found')

test_records = (
    load_or_infer(test_df, 'test', allow_calls=RUN_TEST_API_CALLS)
    if test_df is not None else pd.DataFrame()
)

print('Available direct dev predictions:', len(dev_records), '/', len(dev_df))
print('Available direct test predictions:', len(test_records), '/', 0 if test_df is None else len(test_df))


In [ ]:
def safe_spearman(gold, prediction) -> float:
    value = spearmanr(np.asarray(gold, float), np.asarray(prediction, float)).statistic
    return float(value) if np.isfinite(value) else np.nan

if len(dev_records) == len(dev_df):
    dev_predictions = dev_records[['instance_id', 'CRAI_CEA']].rename(
        columns={'instance_id': 'id', 'CRAI_CEA': 'prediction'}
    )
    dev_evaluation = dev_df[['id', 'CRAI_CEA']].merge(
        dev_predictions, on='id', how='inner', validate='one_to_one'
    )
    if len(dev_evaluation) != len(dev_df):
        raise ValueError('Development prediction/gold row mismatch')

    dev_metrics = pd.DataFrame([{
        'system': 'Direct few-shot CEA',
        'spearman': safe_spearman(
            dev_evaluation['CRAI_CEA'], dev_evaluation['prediction']
        ),
        'mae': mean_absolute_error(
            dev_evaluation['CRAI_CEA'], dev_evaluation['prediction']
        ),
        'n': len(dev_evaluation),
    }])
    display(dev_metrics.round(4))
    dev_metrics.to_csv(
        OUTPUT_DIR / 'cea_direct_fewshot_dev_metrics.tsv', sep='\t', index=False
    )
else:
    print(f'Dev incomplete: {len(dev_records)}/{len(dev_df)} rows')


In [ ]:
if len(dev_records) == len(dev_df):
    dev_export = dev_df[['id']].merge(
        dev_records[['instance_id', 'CRAI_CEA']].rename(columns={'instance_id': 'id'}),
        on='id', how='left', validate='one_to_one'
    )
    if dev_export['CRAI_CEA'].isna().any():
        raise ValueError('Missing dev prediction')
    dev_path = OUTPUT_DIR / 'cea_direct_fewshot_dev_predictions.tsv'
    dev_export.to_csv(dev_path, sep='\t', index=False)
    print('Saved:', dev_path)
else:
    print('Dev export pending complete cache.')

if test_df is not None and len(test_records) == len(test_df):
    test_export = test_df[['id']].merge(
        test_records[['instance_id', 'CRAI_CEA']].rename(columns={'instance_id': 'id'}),
        on='id', how='left', validate='one_to_one'
    )
    if test_export['CRAI_CEA'].isna().any():
        raise ValueError('Missing test prediction')
    test_path = OUTPUT_DIR / 'cea_direct_fewshot_test_predictions.tsv'
    test_export.to_csv(test_path, sep='\t', index=False)
    print('Saved:', test_path)
else:
    print('Test export pending or test split unavailable.')
